Preparing dataframes for experiments

In [ ]:
# This cell prepares the data to be used, run it before all other cells
import geopandas as gpd
import pandas as pd
from shapely.ops import unary_union, linemerge, snap

bikepaths1_GDF = gpd.read_file('.\\data\\bikelanes_updated\\Ciclorrotas.shp')
bikepaths2_GDF = gpd.read_file('.\\data\\bikelanes_updated\\Ciclovias.shp')
bikepaths_GDF = pd.concat([bikepaths1_GDF, bikepaths2_GDF], ignore_index = True)
bikepaths_GDF = bikepaths_GDF.set_crs(epsg=4326)
bikepaths_GDF = bikepaths_GDF.to_crs(epsg=31983)

distances_dic = {'name':[], 'CET_distance':[], 'difference_ratio':[]}
totally_alright_paths = bikepaths_GDF.dissolve(by = 'programa')
totally_alright_paths = totally_alright_paths.reset_index()
unique_bikepaths_GDF = bikepaths_GDF.drop_duplicates(subset = 'programa', keep = 'first')
unique_bikepaths_GDF = unique_bikepaths_GDF.drop(columns='geometry').merge(
    totally_alright_paths[['programa', 'geometry']],
    on = 'programa',
    how = 'left'
)
distances_dic['name'].extend(unique_bikepaths_GDF['programa'])
distances_dic['CET_distance'].extend(unique_bikepaths_GDF['extensao_c'])
distances_dic['difference_ratio'] = [None for _ in distances_dic['name']]
distances_dic['calculated_distance'] = [None for _ in distances_dic['name']]

distances_GDF = gpd.GeoDataFrame(distances_dic, geometry=unique_bikepaths_GDF['geometry'])

for idx, row in distances_GDF.iterrows():
    calculated_distance = bikepaths_GDF[bikepaths_GDF['programa'] == row['name']].length.sum()
    distances_GDF.at[idx, 'calculated_distance'] = calculated_distance
    distances_GDF.at[idx, 'difference_ratio'] = calculated_distance/distances_GDF.at[idx, 'CET_distance']

C:\Users\João Rahal\AppData\Local\Temp\ipykernel_6232\3316277407.py:31: RuntimeWarning: divide by zero encountered in scalar divide
  distances_GDF.at[idx, 'difference_ratio'] = calculated_distance/distances_GDF.at[idx, 'CET_distance']
C:\Users\João Rahal\AppData\Local\Temp\ipykernel_6232\3316277407.py:31: RuntimeWarning: divide by zero encountered in scalar divide
  distances_GDF.at[idx, 'difference_ratio'] = calculated_distance/distances_GDF.at[idx, 'CET_distance']


EXPERIMENT 1:
tried snapping and linemerging routes, seemingly no relevant effect on precision values

In [3]:
for idx, row in distances_GDF.iterrows():
    # possibly_redundant_paths = unary_union(bikepaths_GDF[bikepaths_GDF['programa'] == row['name']]['geometry'])
    possibly_redundant_paths = unary_union(bikepaths_GDF[bikepaths_GDF['programa'] == row['name']]['geometry'].simplify(5))
    snapped_paths = snap(possibly_redundant_paths, possibly_redundant_paths, tolerance = 10) 
    # cleaned_paths = linemerge(snapped_paths)
    distances_GDF.at[idx, 'calculated_distance'] = snapped_paths.length
    # distances_GDF.at[idx, 'difference_ratio'] = cleaned_paths.length/distances_GDF.at[idx, 'CET_distance']

print('projection:', round(distances_GDF['calculated_distance'].sum()), 
      '\t individual sum:', bikepaths_GDF['extensao_t'].sum(), 
      '\t CET official:', distances_GDF['CET_distance'].sum())

projection: 987376 	 individual sum: 996487 	 CET official: 771556


EXPERIMENT 2: official bikelane values and individual lanes' lenght sum show meaningful difference

In [36]:
value_list = []
for idx, row in unique_bikepaths_GDF.iterrows():
    value = bikepaths_GDF[bikepaths_GDF['programa'] == row['programa']]['extensao_t'].sum()
    value_list.append(value)
unique_bikepaths_GDF['measured_length'] = value_list
unique_bikepaths_GDF['difference'] = unique_bikepaths_GDF['measured_length'] - unique_bikepaths_GDF['extensao_c']

unique_bikepaths_GDF.sort_values(by='difference', ascending=False)

,programa,inauguracao,extensao_t,extensao_c,geometry,measured_length,difference
21,CICLOFAIXA ARICANDUVA,2020-10-23,33,11550,"MULTILINESTRING ((350279.528 7389543.135, 3502...",22458,10908
420,CICLOVIA RIO PINHEIROS,2010-02-27,6521,14738,"MULTILINESTRING ((327424.419 7390413.943, 3274...",22233,7495
18,CICLOFAIXA JACU PESSEGO,2020-07-29,658,7405,"MULTILINESTRING ((351206.311 7399039.898, 3512...",14788,7383
37,CICLOFAIXA SAO MIGUEL,2016-11-12,23,7280,"MULTILINESTRING ((351221.159 7400556.008, 3512...",14046,6766
261,CICLOFAIXA ASSIS RIBEIRO - TRECHO 5,2020-10-27,6061,6061,"MULTILINESTRING ((341687.979 7399194.034, 3416...",12080,6019
...,...,...,...,...,...,...,...
189,CICLOFAIXA NAZARE - TRECHO 1,2016-09-05,600,600,"LINESTRING (335566.891 7391496.779, 335570.073...",600,0
187,CICLOVIA JAGUARE,2016-08-31,39,2001,"MULTILINESTRING ((322078.593 7394109.528, 3220...",2001,0
186,CICLOVIA TEOTONIO VILELA,2016-08-22,2624,4105,"MULTILINESTRING ((325632.74 7372155.687, 32563...",4105,0
185,CICLOFAIXA JOAO RAMALHO,2016-08-05,752,2305,"MULTILINESTRING ((328515.652 7396655.128, 3285...",2305,0


In [ ]:
bikepaths_GDF['inauguracao'] = bikepaths_GDF['inauguracao'].astype('string')

bikepaths_year_dic = {'year':[], 'bikepaths_ingrtd':[], 'errors_gte_ratio':[], 'general_proportion':[], 'error_proportion':[], 'year_score':[]}

error_ratio = 1.5

bikepaths_GDF['year'] = bikepaths_GDF['inauguracao'].str[:4]
year_and_bikelanes = bikepaths_GDF['year'].value_counts().sort_index()

high_errors = distances_GDF[distances_GDF['difference_ratio'] >= error_ratio]['name']
bikepath_errors = bikepaths_GDF[bikepaths_GDF['programa'].isin(high_errors)]
year_and_errors = bikepath_errors['year'].value_counts().sort_index()

bikepaths_year_DF = pd.DataFrame(data = {
    'bikepaths_ingrtd': year_and_bikelanes,
    'errors_gte_ratio': year_and_errors.fillna(0)
})

total_bikelanes = bikepaths_year_DF['bikepaths_ingrtd'].sum()
total_errors = bikepaths_year_DF['errors_gte_ratio'].sum()
bikepaths_year_DF['general_proportion'] = bikepaths_year_DF['bikepaths_ingrtd']/total_bikelanes
bikepaths_year_DF['error_proportion'] = bikepaths_year_DF['errors_gte_ratio']/total_errors
bikepaths_year_DF['year_score'] = bikepaths_year_DF['errors_gte_ratio']/bikepaths_year_DF['bikepaths_ingrtd']

display(bikepaths_year_DF.sort_values(by = 'error_proportion', ascending = False))

,bikepaths_ingrtd,errors_gte_ratio,general_proportion,error_proportion,year_score
year,,,,,
2016,382,133,0.16544,0.178523,0.348168
2015,421,121,0.18233,0.162416,0.287411
2021,323,120,0.139887,0.161074,0.371517
2020,235,108,0.101776,0.144966,0.459574
2014,421,98,0.18233,0.131544,0.232779
2023,141,49,0.061065,0.065772,0.347518
2012,46,43,0.019922,0.057718,0.934783
2024,128,34,0.055435,0.045638,0.265625
2017,11,11,0.004764,0.014765,1.0
